In [1]:
# Find the patient's name in a medical document: a simple baseline, without machine learning.
#
# The idea:
# 1. Put the words of each page in reading order: line by line, each line from left to right.
# 2. Rule 1: if the page is a form ("Nom: Hugo", "Prénom: Victor"), take the words written after the labels.
# 3. Rule 2: otherwise, find the first "Monsieur", "Madame"... that does not come right after "Dr" or
#    "Docteur", and take the one or two words after it that start with a capital letter.
import json

# Words that come just before the name of a person: "Monsieur Jean DUPONT", "Mme HUGO".
CIVILITIES = ["monsieur", "madame", "mademoiselle", "m.", "mme", "mlle"]

# Words that come just before the name of a doctor.
# In "le docteur Monsieur Nicolas JACQUES", Nicolas JACQUES is the doctor, not the patient.
DOCTOR_TITLES = ["dr", "dr.", "docteur", "pr", "pr.", "professeur", "docteure"]

# Coordinates in the JSON go from 0 to 1 across the page.
# Two words whose tops differ by more than this (0.5% of the page height) are on two different lines.
LINE_GAP = 0.005

In [2]:
def read_words(page):
    """Return the words of a page in reading order: line by line, each line from left to right."""

    # 1. Make a list of (y, x, text) for every word of the page.
    #    y is the top of the word and x its left side.
    words = []
    for word in page["words"]:
        y = word["bbox"]["y_min"]
        x = word["bbox"]["x_min"]
        words.append((y, x, word["text"]))

    # 2. Sort the words from the top of the page to the bottom.
    #    Python sorts tuples by their first item (y), and by the second item (x) when y is equal.
    words_sorted = sorted(words)

    # 3. Cut the sorted words into lines.
    #    The words of one line have almost the same y, so a new line starts only when y jumps
    #    by more than LINE_GAP. We cannot sort by (y, x) alone: on a scanned page, one word of a
    #    line can sit a tiny bit lower than the others and would be read in the wrong place.
    lines = []
    previous_y = 0
    for y, x, text in words_sorted:
        if len(lines) == 0 or y - previous_y > LINE_GAP:
            lines.append([])  # start a new, empty line
        lines[-1].append((x, text))  # add the word to the current line (the last one of the list)
        previous_y = y

    # 4. Sort each line from left to right (by x), and put all the words in one simple list.
    result = []
    for line in lines:
        line.sort()
        for x, text in line:
            result.append(text)
    return result

In [3]:
def find_patient(document):
    """Return (first_name, last_name) of the patient, or None if no name is found."""
    for page in document["pages"]:
        words = read_words(page)

        # Rule 1: a form, like "Nom: Hugo" and "Prénom: Victor".
        # Look at each word. When it is "Nom:" or "Prénom:", keep the word that comes right after it.
        # We stop one word before the end of the list, because the last word has no word after it.
        first_name = None
        last_name = None
        for i in range(len(words) - 1):
            word = words[i].lower()
            if word == "nom:":
                last_name = words[i + 1]
            if word == "prénom:":
                first_name = words[i + 1]
        if last_name is not None:
            return first_name, last_name

        # Rule 2: the first civility ("Monsieur", "Madame"...) that is not right after a doctor title.
        for i in range(len(words)):
            word = words[i].lower()
            if word not in CIVILITIES:
                continue  # not a civility: go to the next word

            # "le Docteur Monsieur Nicolas JACQUES": the word before is a doctor title,
            # so this name is the doctor's. Skip it and keep looking.
            if i >= 1 and words[i - 1].lower() in DOCTOR_TITLES:
                continue

            # Read at most two words after the civility, as long as they start with a capital letter.
            # "Monsieur Jean DUPONT pour" gives ["Jean", "DUPONT"]; "Mme HUGO un jour" gives ["HUGO"].
            name = []
            for j in range(i + 1, i + 3):
                if j >= len(words):
                    break  # the page ends here
                if words[j].lower() in DOCTOR_TITLES:
                    break  # "Monsieur le Docteur Nicolas JACQUES"
                if not words[j][0].isupper():
                    break  # a word in lowercase: the name is over ("Monsieur le Docteur ..." stops at "le")
                name.append(words[j].strip(".,"))  # remove a "." or "," stuck to the word: "DUPONT," -> "DUPONT"

            # One word is a last name ("Mme HUGO"); two words are a first name and a last name ("Jean DUPONT").
            if len(name) == 1:
                return None, name[0]
            if len(name) == 2:
                return name[0], name[1]
            # No capitalized word after the civility (for example "Madame, Monsieur,"): keep looking.

    # Neither rule found a name on any page.
    return None

In [7]:
# Load one document (the challenge's JSON format) and print the result,
# for example ('Jean', 'DUPONT'), or None if no patient was found.
with open("examples/example3.json", encoding="utf-8") as file:
    document = json.load(file)
print(find_patient(document))

('Jean', 'DUPONT')
